In [1]:
import os
import sys

In [2]:
current_dir = os.getcwd()
root_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.insert(0, root_dir)

In [3]:
from embed.embedder import Embedder

In [4]:
embed = Embedder("../embed/models/Xenova/all-MiniLM-L6-v2")

../embed/models/Xenova/all-MiniLM-L6-v2


## Q1. Embedding a query

In [5]:
query = "How does approximate nearest neighbor search work?"

In [6]:
v = embed.encode(query)

In [7]:
print(v[0])

-0.02058203437252893


## Loading the data

In [8]:
from gitsource import GithubRepositoryDataReader

In [9]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

In [10]:
documents = [file.parse() for file in reader.read()]

In [11]:
len(documents)

72

In [12]:
print(documents[0])

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q2. Cosine similarity

In [13]:
cont_to_vect = [doc['content'] for doc in documents if doc['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md'][0]

In [14]:
cont_embedded = embed.encode(cont_to_vect)

In [15]:
print(cont_embedded.shape)
print(v.shape)

(384,)
(384,)


In [16]:
result = cont_embedded.dot(v)

In [17]:
print(result)

0.36107026789538205


## Q3. Chuncking and search by hand

In [18]:
import numpy as np
from gitsource import chunk_documents

In [19]:
chunks = chunk_documents(documents, size=2000, step=1000)

In [20]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [21]:
len(chunks)

295

In [22]:
X = np.array([embed.encode(chunk['content']) for chunk in chunks])

In [23]:
print(X.shape)

(295, 384)


In [24]:
scores = X.dot(v)

In [25]:
top_score = np.argmax(scores)

In [26]:
top_score

np.int64(94)

In [27]:
chunks[94]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

## Q4. Vector search with minsearch

In [28]:
from minsearch import VectorSearch

In [29]:
vindex = VectorSearch(
    keyword_fields=['filename']
)

In [30]:
vindex.fit(X, chunks)

In [31]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

In [32]:
vindex.search(
    query_vector,
    num_results=3
)

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

## Q5. Text search VS Vector search

In [33]:
from minsearch import Index

In [34]:
index = Index(
    text_fields=['content']
)

index.fit(chunks)

In [35]:
query = "How do I store vectors in PostgreSQL?"
query_vector = embed.encode(query)

In [36]:
results_from_index = index.search(
    query,
    num_results=5
)

In [37]:
results_from_vindex = vindex.search(
    query_vector,
    num_results=5
)

In [38]:
filename_from_res_index = set([doc['filename'] for doc in results_from_index])
filename_from_res_vindex = set([doc['filename'] for doc in results_from_vindex])

In [39]:
filename_from_res_vindex.difference(filename_from_res_index)

{'02-vector-search/lessons/08-pgvector.md'}

## Q6. Hybrid search

In [46]:
def rrf(results_list, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in results_list:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    
        ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [47]:
query = "How do I give the model access to tools?"
query_vector = embed.encode(query)

In [48]:
results_from_index = index.search(
    query,
    num_results=5
)

In [49]:
results_from_vindex = vindex.search(
    query_vector,
    num_results=5
)

In [50]:
print(results_from_index[0])

{'start': 0, 'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can ca

In [51]:
results_text_search_after_rrf = rrf([results_from_index, results_from_vindex]) 

In [53]:
print(results_text_search_after_rrf[0])

{'start': 4000, 'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function cal

In [55]:
print(results_text_search_after_rrf[0]['filename'])

01-agentic-rag/lessons/13-function-calling.md
